## Instalando bibliotecas

In [37]:
!pip install transformers datasets peft accelerate torch

## Importando bibliotecas

In [38]:
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

## Preparando Dataset

In [ ]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='../dataset.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 80
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 20
    })
})


## Carregar o Modelos Pré-Treinado e o Tokenizador

In [40]:
model_name = "unicamp-dl/ptt5-v2-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name,torch_dtype=torch.float16)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Modelo carregado: unicamp-dl/ptt5-v2-large


## Testando Modelo Crú

In [41]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?
Resposta base: : Qual é o local de trabalho? Instruction: Qual é a prioridade? Instruction: Instruction: Instruction: Instruction: In How??:


## Tokenização do Dataset

In [42]:
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["Instruction"],
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["Output"],
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 80
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 20
    })
})


## Preparar o Modelo para LoRA

In [43]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## Verificando Camadas Target

In [44]:
import torch.nn as nn

for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

encoder.block.0.layer.0.SelfAttention.q
encoder.block.0.layer.0.SelfAttention.k
encoder.block.0.layer.0.SelfAttention.v
encoder.block.0.layer.0.SelfAttention.o
encoder.block.0.layer.1.DenseReluDense.wi
encoder.block.0.layer.1.DenseReluDense.wo
encoder.block.1.layer.0.SelfAttention.q
encoder.block.1.layer.0.SelfAttention.k
encoder.block.1.layer.0.SelfAttention.v
encoder.block.1.layer.0.SelfAttention.o
encoder.block.1.layer.1.DenseReluDense.wi
encoder.block.1.layer.1.DenseReluDense.wo
encoder.block.2.layer.0.SelfAttention.q
encoder.block.2.layer.0.SelfAttention.k
encoder.block.2.layer.0.SelfAttention.v
encoder.block.2.layer.0.SelfAttention.o
encoder.block.2.layer.1.DenseReluDense.wi
encoder.block.2.layer.1.DenseReluDense.wo
encoder.block.3.layer.0.SelfAttention.q
encoder.block.3.layer.0.SelfAttention.k
encoder.block.3.layer.0.SelfAttention.v
encoder.block.3.layer.0.SelfAttention.o
encoder.block.3.layer.1.DenseReluDense.wi
encoder.block.3.layer.1.DenseReluDense.wo
encoder.block.4.layer.0.

## Configurar e Injetar LoRA

In [45]:
!pip install --upgrade torchao

In [46]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["q","k","v","o","wi","wo"],  # camadas alvo
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 17,301,504 || all params: 754,969,600 || trainable%: 2.2917


## Data Collator (Modelagem Sequencial)

In [47]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model
)

## Argumentos de Treinamento (Preparando Hiperparâmetros)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/modelo_treinado_unicamp",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=10,
    predict_with_generate=True,
    report_to="none"
)

## Iniciar Treino

In [49]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator
)

## Treinar Modelo

In [50]:
trainer.train()

Step,Training Loss,Validation Loss
100,1.727935,0.794211
200,0.651712,0.472146
300,0.574417,0.427493
400,0.558412,0.415849


TrainOutput(global_step=400, training_loss=5.184995490312576, metrics={'train_runtime': 653.4976, 'train_samples_per_second': 2.448, 'train_steps_per_second': 0.612, 'total_flos': 1774558013030400.0, 'train_loss': 5.184995490312576, 'epoch': 20.0})

## Salvar o Modelo Ajustado e o Tokenizador

In [ ]:
model.save_pretrained("/lora_unicamp_sequencial_finetuned_model")
tokenizer.save_pretrained("/google_unicamp_tokenizer")

('/content/drive/MyDrive/Colab Notebooks/google_unicamp_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/google_unicamp_tokenizer/tokenizer.json')

## Teste pós fine-tuning

In [ ]:

# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("/lora_unicamp_sequencial_finetuned_model")
finetuned_tokenizer = AutoTokenizer.from_pretrained("/unicamp_tokenizer")


# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/768 [00:00<?, ?it/s]

In [55]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?
Resposta ajustada: s e verificações de manutenção quando na. deva e a de.,mente, equipamento de equipamentos,
